# Problem 1
Develop an effective BoW representation plus binary Logistic Regression classifier pipeline, aiming to produce the best possible performance on heldout data.

- x_train_df - from x_train.csv, training input data (passages and associated features), each row = 1 passge of text
- y_train_df - from y_train.csv, training labels (correct reading level category for each passage), use to teach model what to predict
- x_test_df - from x_test.csv, test input data (passages we'll be making predictions for)

In [ ]:
# loading the data

import pandas as pd

x_train_df = pd.read_csv('data_readinglevel/x_train.csv')
y_train_df = pd.read_csv('data_readinglevel/y_train.csv')
x_test_df = pd.read_csv('data_readinglevel/x_test.csv')

# extract "text" column from training and test data
tr_list_of_text = x_train_df['text'].values.tolist()
te_list_of_text = x_test_df['text'].values.tolist()

# extract correct training labels ("coarse label")
y_train = y_train_df['Coarse Label'].values

print("Training examples:", len(tr_list_of_text))
print("Training labels:", len(y_train))
print("Test examples:", len(te_list_of_text))

# how many training examples belong in each class (are the two classes balanced?)
print(y_train_df['Coarse Label'].value_counts())

training examples: 5557
training labels: 5557
test examples: 1197
Coarse Label
Key Stage 4-5    3048
Key Stage 2-3    2509
Name: count, dtype: int64


In [29]:
# build a baseline model 
# this is the simplest version of the required model (BoW + logistic regression)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# custom_pattern = r"(?u)\b\w+[\'\w]*\b"

# create reusable pipeline function
def create_model(min_df=1, binary=False, stop_words=None, C=1.0):
  return Pipeline([
    # ("vectorizer", CountVectorizer(min_df=min_df, binary=binary, stop_words=stop_words, token_pattern=custom_pattern)),
    ("vectorizer", CountVectorizer(min_df=min_df, binary=binary, stop_words=stop_words)),
    ("classifier", LogisticRegression(C=C, max_iter=1000))
  ])

# logistic regression learns how numerical features from CountVectorizer relate to the labels
base_model = create_model()

In [27]:
# evaluate the baseline model with cross-validation
# benchmark to compare against preprocessing experiments

import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# create a cross-validation function
def evaluate_model(model, x, y, cv):
  results = cross_validate(model, x, y, cv=cv, scoring="roc_auc", return_train_score=True)

  print(f"Mean training AUROC: {results['train_score'].mean():.4f}")
  print(f"Mean validation AUROC: {results['test_score'].mean():.4f}")
  print(f"Validation AUROC std: {results['test_score'].std():.4f}")
  print("\n")

  return results

base_results = evaluate_model(base_model, tr_list_of_text, y_train, cv)

Mean training AUROC: 1.0000
Mean validation AUROC: 0.8115
Validation AUROC std: 0.0090




Baseline model fits the training data almost perfectly, but is worse on unseen validation data

In [34]:
# experiment with BoW preprocessing!

model_min_df2 = create_model(min_df=2) # removing rare words (appear fewer than 2 docs)
model_min_df5 = create_model(min_df=5) # remove even more rare words (appear fewer than 5 docs)
model_binary = create_model(binary=True) # record whether word appears, instead of how often
model_no_stopwords = create_model(stop_words="english") # remove common english words

# evaluate each model

results_min_df2 = evaluate_model(model_min_df2, tr_list_of_text, y_train, cv)
results_min_df5 = evaluate_model(model_min_df5, tr_list_of_text, y_train, cv)
results_binary = evaluate_model(model_binary, tr_list_of_text, y_train, cv)
results_no_stopwords = evaluate_model(model_no_stopwords, tr_list_of_text, y_train, cv)

# best score: 0.8223

Mean training AUROC: 1.0000
Mean validation AUROC: 0.8139
Validation AUROC std: 0.0108


Mean training AUROC: 0.9999
Mean validation AUROC: 0.7975
Validation AUROC std: 0.0084


Mean training AUROC: 1.0000
Mean validation AUROC: 0.8075
Validation AUROC std: 0.0094


Mean training AUROC: 1.0000
Mean validation AUROC: 0.8223
Validation AUROC std: 0.0106




In [ ]:
# fit the final model with chosen BoW configuration

final_model = create_model(stop_words="english")
final_model.fit(tr_list_of_text, y_train)

vocab_size = len(final_model.named_steps["vectorizer"].vocabulary_)
print("Final vocab size:", vocab_size)